# L18 — Non-Stationary Arrivals and Batch Processes

**Module**: M06 | **Chapter**: 8 | **Lecture**: L18

## Learning Objectives
By the end of this notebook you will be able to:
1. Simulate a non-homogeneous Poisson process (NHPP) using the thinning algorithm.
2. Model time-varying arrival rates with an hourly rate table.
3. Implement batch arrivals (groups of entities arriving simultaneously).
4. Identify the danger of using steady-state formulas for time-varying systems.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist

## 1. Non-Homogeneous Poisson Process via Thinning

When the arrival rate varies with time — λ(t) — the inter-arrival times are no longer exponential.

**Lewis-Shedler thinning algorithm**:
1. Let λ* = max{λ(t)} be the maximum rate over the simulation horizon
2. Generate candidate arrivals from a Poisson process with rate λ*
3. Accept each candidate at time t with probability λ(t)/λ*; reject otherwise
4. Accepted candidates form the NHPP

In [ ]:
# Clinic hourly arrival rate profile (patients/hr)
# Represents a typical walk-in clinic with morning and afternoon peaks
RATE_TABLE = {
    0: 2.0, 1: 1.0, 2: 0.5, 3: 0.5, 4: 0.5, 5: 1.0,
    6: 3.0, 7: 6.0, 8: 9.0, 9: 10.0, 10: 9.0, 11: 8.0,
    12: 6.0, 13: 7.0, 14: 9.0, 15: 10.0, 16: 8.0, 17: 7.0,
    18: 5.0, 19: 4.0, 20: 3.0, 21: 2.5, 22: 2.0, 23: 2.0,
}

def rate_fn(t: float) -> float:
    """Return arrival rate (patients/hr) at simulation time t (hr)."""
    hour = int(t) % 24
    return RATE_TABLE[hour]

# Visualise the rate profile
hours = np.arange(0, 24, 0.1)
rates = [rate_fn(h) for h in hours]

fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(hours, rates, alpha=0.3, color='steelblue')
ax.plot(hours, rates, color='steelblue', lw=2)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Arrival rate (patients/hr)')
ax.set_title('Non-stationary arrival rate profile (clinic example)')
ax.set_xticks(range(0, 25, 2))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def nhpp_thinning_arrivals(env, rate_fn, lam_star, rng):
    """
    Generator that yields (via env.process) NHPP arrival times.
    Uses Lewis-Shedler thinning with maximum rate lam_star.
    """
    while True:
        # Generate candidate from homogeneous Poisson(lam_star)
        yield env.timeout(rng.exponential(1.0 / lam_star))
        # Accept with probability lambda(t)/lambda*
        if rng.random() < rate_fn(env.now) / lam_star:
            yield simpy.Event(env).succeed()  # signal arrival


def nhpp_simulation(rate_fn, lam_star, mu, c,
                    sim_time=24*30, seed=0):   # 30 days
    """Simulate M_t/M/c queue with time-varying NHPP arrivals."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=c)
    records = []

    def patient():
        arrival = env.now
        with server.request() as req:
            yield req
            wait = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu))
        records.append({'hour': arrival % 24, 'wait': wait})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam_star))
            if rng.random() < rate_fn(env.now) / lam_star:
                env.process(patient())

    env.process(arrivals())
    env.run(until=sim_time)
    return pd.DataFrame(records)


lam_star = max(RATE_TABLE.values())  # = 10.0
mu_clinic, c_clinic = 6.0, 2   # 2 nurses, each serving 6 patients/hr

df = nhpp_simulation(rate_fn, lam_star, mu_clinic, c_clinic, seed=42)
print(f"Simulated arrivals: {len(df):,} over 30 days")
print(f"Overall mean wait: {df['wait'].mean()*60:.2f} min")

In [ ]:
# Wait by hour of day
hourly = df.groupby('hour')['wait'].agg(['mean', 'count'])
hourly['mean_min'] = hourly['mean'] * 60

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Arrival rate
ax1.bar(range(24), [RATE_TABLE[h] for h in range(24)],
        color='steelblue', alpha=0.6, label='λ(t) (patients/hr)')
ax1.set_ylabel('Arrival rate (patients/hr)')
ax1.legend(fontsize=8)

# Mean wait
ax2.bar(hourly.index, hourly['mean_min'],
        color='tomato', alpha=0.7, label='Mean wait (min)')
ax2.set_xlabel('Hour of day')
ax2.set_ylabel('Mean wait (min)')
ax2.legend(fontsize=8)

plt.suptitle(f'Non-stationary clinic: c={c_clinic} nurses, μ={mu_clinic}/hr')
plt.tight_layout()
plt.show()

print(f"Peak wait hour: {hourly['mean_min'].idxmax()} (mean wait = {hourly['mean_min'].max():.1f} min)")

## 2. Danger of Steady-State Approximation for NHPP

If you naively apply the steady-state M/M/c formula at the peak rate, you get a conservative (worst-case) estimate. At the off-peak rate, you underestimate waiting. Neither is the simulation truth.

In [ ]:
from math import factorial

def erlang_c_wq(c, lam, mu):
    rho = lam / (c * mu)
    if rho >= 1.0:
        return float('inf')
    a = lam / mu
    s = sum(a**n / factorial(n) for n in range(c))
    last = (a**c / factorial(c)) / (1 - rho)
    C = last / (s + last)
    return C / (c * mu - lam)

# Steady-state predictions for peak (10/hr) and off-peak (2/hr)
for lam_approx, label in [(10.0, 'Peak λ=10'), (2.0, 'Off-peak λ=2'),
                           (df['hour'].map(RATE_TABLE).mean(), 'Mean λ')]:
    wq_ss = erlang_c_wq(c_clinic, lam_approx, mu_clinic) * 60
    print(f"  {label}: steady-state Wq = {wq_ss:.2f} min")

print(f"\n  Simulation overall mean Wq = {df['wait'].mean()*60:.2f} min")
print()
print("The steady-state formula cannot capture time-of-day variation.")
print("Simulation is the only rigorous approach for NHPP systems.")

## 3. Batch Arrivals

Sometimes entities arrive in groups (family clinic, tour buses, production batches).  
The batch size can be random.

In [ ]:
def batch_arrival_sim(lam_batch: float, batch_size_fn,
                      mu: float, c: int,
                      sim_time: float = 100_000, seed: int = 0) -> dict:
    """
    Queue with batch Poisson arrivals.

    lam_batch    : rate of batch arrival events
    batch_size_fn: callable(rng) -> int, returns number of entities per batch
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=c)
    waits = []

    def one_customer():
        arrival = env.now
        with server.request() as req:
            yield req
            waits.append(env.now - arrival)
            yield env.timeout(rng.exponential(1.0 / mu))

    def batch_arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam_batch))
            batch_size = batch_size_fn(rng)
            for _ in range(batch_size):
                env.process(one_customer())

    env.process(batch_arrivals())
    env.run(until=sim_time)

    lam_individual = lam_batch * batch_size_fn.__doc_mean__ if hasattr(batch_size_fn, '__doc_mean__') else None
    return {'Wq': np.mean(waits), 'n_served': len(waits)}


# Compare: Poisson(lam=5) vs. batches of {1,2,3} arriving at lam_batch=5/2
lam_ind = 5.0
mu_s, c_s = 7.0, 1

# Individual (standard Poisson)
def unit_batch(rng):
    return 1
r_individual = batch_arrival_sim(lam_ind, unit_batch, mu_s, c_s, seed=0)

# Geometric batch size with mean 2
def geo_batch(rng):
    return int(rng.geometric(p=0.5))  # mean = 2
r_batch = batch_arrival_sim(lam_ind/2, geo_batch, mu_s, c_s, seed=0)

print(f"Individual arrivals (λ=5, μ=7, c=1): Wq = {r_individual['Wq']:.4f} (served: {r_individual['n_served']:,})")
print(f"Batch arrivals (λ_batch=2.5, E[B]=2): Wq = {r_batch['Wq']:.4f} (served: {r_batch['n_served']:,})")
print()
print("Same effective individual rate, but batch arrivals cause MORE waiting")
print("because multiple customers land simultaneously and create bursts.")

---
## Try It Yourself

1. **Piecewise-constant rate**: Simplify the rate function to three periods: off-peak (λ=2, 0–8 hr), peak (λ=10, 8–18 hr), evening (λ=4, 18–24 hr). Implement this as a piecewise-constant thinning function and compare simulation results to a period-by-period steady-state approximation. For which period is the approximation most wrong?

2. **Time-based staffing**: The clinic schedules c=1 nurse in off-peak hours and c=3 at peak. Modify `nhpp_simulation` to accept a `c_fn(t)` that returns the number of servers at time t. Change the `server._capacity` dynamically during the simulation. What is the mean wait at peak hours now?

3. **Batch service**: Some systems serve customers in batches (a bus picks up N passengers, a batch reactor processes B items). Implement batch service: the server waits until either B customers are queued or a timeout of T seconds elapses, then serves all waiting customers simultaneously. Compare throughput and mean wait to individual service.